# Session 9: Distance-Based Methods — KNN & K-Means

This notebook covers:
1. **Feature Space & Distance** — visualizing data as points in space
2. **K-Nearest Neighbors** — manual walkthrough, then sklearn
3. **K-Means Clustering** — algorithm, choosing K, limitations
4. **Exercises**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.datasets import load_iris, make_blobs, make_moons, make_circles, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             silhouette_score, silhouette_samples)

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)

---
# Section 1: Feature Space & Distance

Every row in a dataset is a point in space. Each feature is a dimension.

In [ ]:
# Load the Iris dataset — 4 numeric features, 3 classes
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target

colors = {0: '#4C72B0', 1: '#DD8452', 2: '#55A868'}
df.head()

### 2 features → 2D scatter plot
Each row becomes a point. The axes are the features.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for label, color in colors.items():
    mask = df['species'] == label
    ax.scatter(df.loc[mask, 'petal length (cm)'],
               df.loc[mask, 'petal width (cm)'],
               c=color, label=iris.target_names[label],
               s=60, alpha=0.7, edgecolors='white', linewidth=0.5)
ax.set_xlabel('Petal Length (cm)', fontsize=13)
ax.set_ylabel('Petal Width (cm)', fontsize=13)
ax.set_title('2 Features → 2D Feature Space', fontsize=15)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### All 6 pairwise views into 4D space
We have 4 features, but can only plot 2 at a time. Each subplot is a different "window" into the full 4D space.

In [ ]:
from itertools import combinations

feature_names = iris.feature_names
short_names = [n.replace(' (cm)', '').replace('sepal ', 'S. ').replace('petal ', 'P. ')
               for n in feature_names]

feature_pairs = list(combinations(range(4), 2))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for idx, (f0, f1) in enumerate(feature_pairs):
    ax = axes[idx]
    for label, color in colors.items():
        mask = df['species'] == label
        ax.scatter(df.iloc[mask.values, f0], df.iloc[mask.values, f1],
                   c=color, label=iris.target_names[label],
                   s=40, alpha=0.7, edgecolors='white', linewidth=0.5)
    ax.set_xlabel(short_names[f0], fontsize=11)
    ax.set_ylabel(short_names[f1], fontsize=11)
    ax.set_title(f'{short_names[f0]} vs {short_names[f1]}', fontsize=12)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle('6 Pairwise Views into 4D Feature Space', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Distance between points
Once we see data as points in space, we can measure how far apart they are.

In [ ]:
from scipy.spatial.distance import euclidean, cityblock, cosine

# Pick two flowers from different species
p1 = iris.data[0]   # setosa
p2 = iris.data[100]  # virginica

print(f"Setosa:    {dict(zip(feature_names, p1))}")
print(f"Virginica: {dict(zip(feature_names, p2))}")
print()
print(f"Euclidean distance: {euclidean(p1, p2):.3f}")
print(f"Manhattan distance: {cityblock(p1, p2):.3f}")
print(f"Cosine distance:    {cosine(p1, p2):.3f}")

---
# Section 2: K-Nearest Neighbors

The simplest ML algorithm: store the training data, and when a new point arrives, look at its nearest neighbors.

## 2.1 KNN by Hand
Let's walk through the algorithm step by step with a tiny 2D dataset.

In [ ]:
# Training data: 10 points, 2 features, 2 classes
X_train_manual = np.array([
    [1.0, 3.0],  [2.0, 4.5],  [1.5, 1.5],  [3.0, 3.5],  [2.5, 2.0],
    [6.0, 5.0],  [5.5, 3.5],  [7.0, 4.0],  [6.5, 2.0],  [5.0, 5.5],
])
y_train_manual = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1])

# The new point we want to classify
query = np.array([4.0, 3.75])

class_colors = {0: '#4C72B0', 1: '#DD8452'}
class_markers = {0: 'o', 1: '^'}

def plot_base(ax, show_query=True, title=''):
    for cls in [0, 1]:
        mask = y_train_manual == cls
        ax.scatter(X_train_manual[mask, 0], X_train_manual[mask, 1],
                   c=class_colors[cls], marker=class_markers[cls],
                   s=100, edgecolors='black', linewidth=1, zorder=3,
                   label=f'Class {cls}')
    if show_query:
        ax.scatter(*query, c='red', marker='*', s=300, edgecolors='black',
                   linewidth=1.5, zorder=5, label='Query point')
    ax.set_xlabel('$x_1$', fontsize=13)
    ax.set_ylabel('$x_2$', fontsize=13)
    ax.set_xlim(-0.5, 8.5)
    ax.set_ylim(-0.5, 7)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=10, loc='upper left')
    if title:
        ax.set_title(title, fontsize=14)

fig, ax = plt.subplots(figsize=(8, 6))
plot_base(ax, show_query=True, title='Training Data + Query Point')
plt.tight_layout()
plt.show()

### Step 1: Compute distance from the query to every training point

In [ ]:
distances = np.sqrt(np.sum((X_train_manual - query) ** 2, axis=1))

print(f"Query point: ({query[0]}, {query[1]})\n")
print(f"{'Point':>6}  {'Location':>12}  {'Class':>7}  {'Distance':>10}")
print(f"{'─'*6}  {'─'*12}  {'─'*7}  {'─'*10}")
for i in range(len(X_train_manual)):
    loc = f"({X_train_manual[i,0]:.1f}, {X_train_manual[i,1]:.1f})"
    print(f"  {i+1:>3}   {loc:>12}  {'Class '+str(y_train_manual[i]):>7}  {distances[i]:>10.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
plot_base(ax, show_query=True, title='Compute Distances')

for i in range(len(X_train_manual)):
    ax.plot([query[0], X_train_manual[i, 0]], [query[1], X_train_manual[i, 1]],
            color='gray', linewidth=0.8, alpha=0.5, zorder=1)
    mid = (query + X_train_manual[i]) / 2
    ax.text(mid[0], mid[1], f'{distances[i]:.2f}', fontsize=8,
            ha='center', va='center', color='#555555',
            bbox=dict(boxstyle='round,pad=0.15', facecolor='white',
                      edgecolor='none', alpha=0.8))

plt.tight_layout()
plt.show()

### Step 2: Sort by distance and pick K=3 nearest

In [ ]:
sorted_indices = np.argsort(distances)
k = 3

print(f"Training points sorted by distance:\n")
print(f"{'Rank':>4}  {'Point':>6}  {'Location':>12}  {'Class':>7}  {'Distance':>10}")
print(f"{'─'*4}  {'─'*6}  {'─'*12}  {'─'*7}  {'─'*10}")
for rank, i in enumerate(sorted_indices):
    loc = f"({X_train_manual[i,0]:.1f}, {X_train_manual[i,1]:.1f})"
    marker = " ◄── nearest 3" if rank < k else ""
    print(f"  {rank+1:>2}   {i+1:>4}   {loc:>12}  {'Class '+str(y_train_manual[i]):>7}  {distances[i]:>10.3f}{marker}")

In [ ]:
neighbor_indices = sorted_indices[:k]

fig, ax = plt.subplots(figsize=(8, 6))
plot_base(ax, show_query=True, title=f'K={k} Nearest Neighbors')

# Draw lines to the k nearest neighbors
for rank, i in enumerate(neighbor_indices):
    color = class_colors[y_train_manual[i]]
    ax.plot([query[0], X_train_manual[i, 0]], [query[1], X_train_manual[i, 1]],
            color=color, linewidth=2, alpha=0.8, zorder=2)
    mid = (query + X_train_manual[i]) / 2
    ax.text(mid[0], mid[1], f'#{rank+1}: d={distances[i]:.2f}',
            fontsize=9, ha='center', va='center', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor=color, alpha=0.9))

# Neighborhood radius circle
radius = distances[neighbor_indices[-1]]
circle = plt.Circle(query, radius, fill=False, color='red',
                     linestyle='--', linewidth=1.5, alpha=0.5)
ax.add_patch(circle)

# Fade non-neighbors
for i in range(len(X_train_manual)):
    if i not in neighbor_indices:
        ax.scatter(X_train_manual[i, 0], X_train_manual[i, 1],
                   c='lightgray', marker=class_markers[y_train_manual[i]],
                   s=100, edgecolors='gray', linewidth=0.5, zorder=2)

plt.tight_layout()
plt.show()

### Step 3: Majority vote → prediction

In [ ]:
neighbor_labels = y_train_manual[neighbor_indices]
unique, counts = np.unique(neighbor_labels, return_counts=True)

print(f"K = {k} nearest neighbors:\n")
for rank, i in enumerate(neighbor_indices):
    print(f"  Neighbor #{rank+1}: Point {i+1} at ({X_train_manual[i,0]:.1f}, {X_train_manual[i,1]:.1f})"
          f"  →  Class {y_train_manual[i]}  (distance = {distances[i]:.3f})")

print(f"\nVote tally:")
for cls, count in zip(unique, counts):
    bar = '█' * (count * 5)
    print(f"  Class {cls}: {count} vote(s)  {bar}")

prediction = unique[np.argmax(counts)]
print(f"\n★ Prediction: Class {prediction}")

### Same data, different K → different predictions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, k_val in zip(axes, [1, 3, 5]):
    neighbor_idx = sorted_indices[:k_val]
    nb_labels = y_train_manual[neighbor_idx]
    unique, counts = np.unique(nb_labels, return_counts=True)
    pred = unique[np.argmax(counts)]

    plot_base(ax, show_query=True, title=f'K = {k_val}')

    for i in neighbor_idx:
        ax.plot([query[0], X_train_manual[i, 0]], [query[1], X_train_manual[i, 1]],
                color=class_colors[y_train_manual[i]], linewidth=2, alpha=0.7, zorder=2)

    radius = distances[neighbor_idx[-1]]
    circle = plt.Circle(query, radius, fill=False, color='red',
                         linestyle='--', linewidth=1.5, alpha=0.5)
    ax.add_patch(circle)

    for i in range(len(X_train_manual)):
        if i not in neighbor_idx:
            ax.scatter(X_train_manual[i, 0], X_train_manual[i, 1],
                       c='lightgray', marker=class_markers[y_train_manual[i]],
                       s=100, edgecolors='gray', linewidth=0.5, zorder=2)

    vote_str = ', '.join([f'Class {c}: {n}' for c, n in zip(unique, counts)])
    ax.text(0.5, -0.08, f'Votes: {vote_str}  →  Predict Class {pred}',
            transform=ax.transAxes, ha='center', fontsize=11,
            fontweight='bold', color=class_colors[pred])

fig.suptitle('Same Data, Different K → Different Predictions', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 2.2 KNN in Scikit-Learn

Now let's do this properly on a real dataset with a pipeline.

In [ ]:
# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Features:     {X_train.shape[1]}")
print(f"Classes:      {np.unique(y_train)}")

### Pipeline: Scale first, then KNN
Distance-based methods are sensitive to feature scale. A pipeline ensures we scale properly (fit on train, transform both).

In [ ]:
# Build a pipeline
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])

# Fit on training data
knn_pipeline.fit(X_train, y_train)

# Evaluate on test data
y_pred = knn_pipeline.predict(X_test)
print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Choosing K with cross-validation

In [ ]:
k_range = range(1, 26)
cv_scores = []

for k in k_range:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

best_k = list(k_range)[np.argmax(cv_scores)]
print(f"Best K: {best_k} (CV accuracy: {max(cv_scores):.3f})")

plt.figure(figsize=(10, 5))
plt.plot(k_range, cv_scores, 'o-', linewidth=2, markersize=6)
plt.axvline(x=best_k, color='red', linestyle='--', alpha=0.7, label=f'Best K={best_k}')
plt.xlabel('K (number of neighbors)', fontsize=13)
plt.ylabel('Cross-Validation Accuracy', fontsize=13)
plt.title('Choosing K with Cross-Validation', fontsize=14)
plt.legend(fontsize=11)
plt.xticks(range(1, 26))
plt.tight_layout()
plt.show()

### Using GridSearchCV with a pipeline
We can tune K, the distance metric, and the weighting scheme all at once.

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan'],
}

grid_search = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")
print(f"Test accuracy:    {grid_search.score(X_test, y_test):.3f}")

### Visualizing the decision boundary (2D slice)
We'll use just 2 features so we can plot the boundary.

In [ ]:
def plot_decision_boundary(X, y, model, title='', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    h = 0.05
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))

    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.15, cmap='viridis')
    ax.contour(xx, yy, Z, colors='black', linewidths=0.5, alpha=0.3)
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis',
                         s=40, edgecolors='black', linewidth=0.5)
    ax.set_title(title, fontsize=13)
    return ax

# Use petal length and petal width only
X_2d = X_train[:, 2:4]
X_2d_test = X_test[:, 2:4]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, k_val in zip(axes, [1, 7, 25]):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k_val))
    ])
    pipe.fit(X_2d, y_train)

    # Plot in original (unscaled) space for interpretability
    plot_decision_boundary(X_2d, y_train, pipe,
                           title=f'K={k_val} (acc={pipe.score(X_2d_test, y_test):.2f})',
                           ax=ax)
    ax.set_xlabel('Petal Length (cm)')
    ax.set_ylabel('Petal Width (cm)')

fig.suptitle('KNN Decision Boundaries: Effect of K', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
# Section 3: K-Means Clustering

K-Means is an unsupervised algorithm: no labels, just data. The goal is to discover groups.

## 3.1 Basic K-Means

In [ ]:
# Generate synthetic data with 4 clusters
X_blobs, y_true_blobs = make_blobs(
    n_samples=500, centers=4, cluster_std=2, random_state=423
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], s=50, alpha=0.7)
axes[0].set_title('Raw Data (No Labels)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true_blobs, cmap='viridis', s=50, alpha=0.7)
axes[1].set_title('True Labels (Hidden from Algorithm)')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

In [ ]:
# Apply K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
y_pred_blobs = kmeans.fit_predict(X_blobs)
centers = kmeans.cluster_centers_

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_pred_blobs, cmap='viridis', s=50, alpha=0.7)
axes[0].scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200,
                edgecolors='black', linewidths=2, label='Centroids')
axes[0].set_title('K-Means Result (K=4)')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].legend()

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true_blobs, cmap='viridis', s=50, alpha=0.7)
axes[1].set_title('True Labels')
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print(f"Inertia (WCSS): {kmeans.inertia_:.2f}")
print(f"Iterations to converge: {kmeans.n_iter_}")

## 3.2 Visualizing K-Means Iterations
The algorithm alternates between assigning points to the nearest centroid and moving centroids to the mean of their assigned points.

In [ ]:
def plot_kmeans_iterations(X, n_clusters=4, max_iter=6):
    np.random.seed(0)
    initial_idx = np.random.choice(len(X), n_clusters, replace=False)
    centroids = X[initial_idx].copy()

    n_cols = 3
    n_rows = (max_iter + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.5 * n_rows))
    axes = axes.flatten()

    for i in range(max_iter):
        # Assignment step
        dists = np.sqrt(((X[:, np.newaxis] - centroids) ** 2).sum(axis=2))
        labels = dists.argmin(axis=1)

        ax = axes[i]
        ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=30, alpha=0.6)
        ax.scatter(centroids[:, 0], centroids[:, 1],
                   c='red', marker='X', s=200, edgecolors='black', linewidths=2)
        ax.set_title(f'Iteration {i + 1}', fontsize=12)
        ax.set_xlabel('Feature 1')
        ax.set_ylabel('Feature 2')

        # Update step
        new_centroids = np.array([
            X[labels == k].mean(axis=0) if (labels == k).sum() > 0 else centroids[k]
            for k in range(n_clusters)
        ])

        if np.allclose(centroids, new_centroids):
            ax.set_title(f'Iteration {i + 1} (Converged!)', fontsize=12)
            for j in range(i + 1, max_iter):
                axes[j].axis('off')
            break

        centroids = new_centroids

    plt.suptitle('K-Means Convergence: Iteration by Iteration', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

plot_kmeans_iterations(X_blobs, n_clusters=4, max_iter=6)

## 3.3 Feature Scaling Matters for K-Means Too

In [ ]:
# Create data with features on very different scales
X_unscaled, y_unscaled = make_blobs(n_samples=500, centers=3, cluster_std=2.5, random_state=42)
X_unscaled[:, 1] = X_unscaled[:, 1] * 2000  # Blow up feature 2

# K-Means without scaling
kmeans_unscaled = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_unscaled = kmeans_unscaled.fit_predict(X_unscaled)

# K-Means with scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_unscaled)
kmeans_scaled = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_scaled = kmeans_scaled.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_unscaled[:, 0], X_unscaled[:, 1], c=labels_unscaled, cmap='viridis', s=50, alpha=0.7)
axes[0].set_title('K-Means WITHOUT Scaling\n(Feature 2 dominates)')
axes[0].set_xlabel('Feature 1 (small scale)')
axes[0].set_ylabel('Feature 2 (large scale)')

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels_scaled, cmap='viridis', s=50, alpha=0.7)
axes[1].set_title('K-Means WITH Scaling\n(Both features contribute equally)')
axes[1].set_xlabel('Feature 1 (standardized)')
axes[1].set_ylabel('Feature 2 (standardized)')

plt.tight_layout()
plt.show()

print(f"Silhouette score WITHOUT scaling: {silhouette_score(X_unscaled, labels_unscaled):.3f}")
print(f"Silhouette score WITH scaling:    {silhouette_score(X_scaled, labels_scaled):.3f}")

## 3.4 Choosing K: Elbow Method and Silhouette Analysis

In [ ]:
# Generate data with 5 true clusters
X_choose, y_true_choose = make_blobs(n_samples=500, centers=5, cluster_std=1.8, random_state=42)

plt.figure(figsize=(6, 5))
plt.scatter(X_choose[:, 0], X_choose[:, 1], s=50, alpha=0.7)
plt.title('Raw Data — How Many Clusters?')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.tight_layout()
plt.show()

In [ ]:
k_range = range(2, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_choose)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_choose, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=5, color='r', linestyle='--', alpha=0.7, label='Elbow at K=5')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method')
axes[0].legend()
axes[0].set_xticks(list(k_range))

axes[1].plot(k_range, sil_scores, 'go-', linewidth=2, markersize=8)
best_k_sil = list(k_range)[np.argmax(sil_scores)]
axes[1].axvline(x=best_k_sil, color='r', linestyle='--', alpha=0.7,
                label=f'Best K={best_k_sil}')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].legend()
axes[1].set_xticks(list(k_range))
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 3.5 Initialization: Random vs K-Means++

In [ ]:
X_init, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)

random_inertias = []
kpp_inertias = []
for seed in range(20):
    km_rand = KMeans(n_clusters=4, init='random', n_init=1, random_state=seed)
    km_rand.fit(X_init)
    random_inertias.append(km_rand.inertia_)

    km_pp = KMeans(n_clusters=4, init='k-means++', n_init=1, random_state=seed)
    km_pp.fit(X_init)
    kpp_inertias.append(km_pp.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].boxplot([random_inertias, kpp_inertias], labels=['Random', 'K-Means++'])
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Inertia by Initialization Method')

axes[1].hist(random_inertias, bins=10, alpha=0.7, label='Random')
axes[1].hist(kpp_inertias, bins=10, alpha=0.7, label='K-Means++')
axes[1].set_xlabel('Inertia (WCSS)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Inertia Histogram')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Random init   — Mean: {np.mean(random_inertias):.1f}, Std: {np.std(random_inertias):.1f}")
print(f"K-Means++ init — Mean: {np.mean(kpp_inertias):.1f}, Std: {np.std(kpp_inertias):.1f}")

## 3.6 When K-Means Fails

In [ ]:
# Non-convex shapes: moons and circles
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)
X_circles, y_circles = make_circles(n_samples=300, factor=0.5, noise=0.05, random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Moons
axes[0, 0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', s=50, alpha=0.7)
axes[0, 0].set_title('True Clusters (Crescents)')

km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_moons = km_moons.fit_predict(X_moons)
axes[0, 1].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_moons, cmap='viridis', s=50, alpha=0.7)
axes[0, 1].scatter(km_moons.cluster_centers_[:, 0], km_moons.cluster_centers_[:, 1],
                   c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[0, 1].set_title('K-Means Result (Fails!)')

# Circles
axes[1, 0].scatter(X_circles[:, 0], X_circles[:, 1], c=y_circles, cmap='viridis', s=50, alpha=0.7)
axes[1, 0].set_title('True Clusters (Concentric Circles)')

km_circles = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_circles = km_circles.fit_predict(X_circles)
axes[1, 1].scatter(X_circles[:, 0], X_circles[:, 1], c=labels_circles, cmap='viridis', s=50, alpha=0.7)
axes[1, 1].scatter(km_circles.cluster_centers_[:, 0], km_circles.cluster_centers_[:, 1],
                   c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[1, 1].set_title('K-Means Result (Fails!)')

for ax in axes.flatten():
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('K-Means Assumes Spherical Clusters', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Different sizes and densities
np.random.seed(42)
cluster1 = np.random.randn(200, 2) * 2.5 + [0, 0]    # Large, sparse
cluster2 = np.random.randn(50, 2) * 0.3 + [6, 6]      # Small, dense
cluster3 = np.random.randn(50, 2) * 0.3 + [6, -2]     # Small, dense

X_unequal = np.vstack([cluster1, cluster2, cluster3])
y_unequal = np.array([0]*200 + [1]*50 + [2]*50)

km_unequal = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_unequal = km_unequal.fit_predict(X_unequal)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(X_unequal[:, 0], X_unequal[:, 1], c=y_unequal, cmap='viridis', s=50, alpha=0.7)
axes[0].set_title('True Clusters (Different Sizes/Densities)')

axes[1].scatter(X_unequal[:, 0], X_unequal[:, 1], c=labels_unequal, cmap='viridis', s=50, alpha=0.7)
axes[1].scatter(km_unequal.cluster_centers_[:, 0], km_unequal.cluster_centers_[:, 1],
                c='red', marker='X', s=200, edgecolors='black', linewidths=2)
axes[1].set_title('K-Means Result (Splits the Large Cluster)')

for ax in axes:
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

---
# Section 4: Exercises

## Exercise 1: KNN on the Wine Dataset

The Wine dataset has 13 features and 3 classes.

1. Load the data with `load_wine()` and do a train/test split (80/20, stratified).
2. Build a pipeline with `StandardScaler` and `KNeighborsClassifier`.
3. Use `GridSearchCV` to search over `n_neighbors` (1 to 20) and `weights` ('uniform', 'distance').
4. Print the best parameters and the test accuracy.

In [ ]:
# Write your code here


In [ ]:
#@title Click to reveal solution
wine = load_wine()
X_w, y_w = wine.data, wine.target

X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
    X_w, y_w, test_size=0.2, random_state=42, stratify=y_w
)

pipe_w = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_grid_w = {
    'knn__n_neighbors': range(1, 21),
    'knn__weights': ['uniform', 'distance'],
}

grid_w = GridSearchCV(pipe_w, param_grid_w, cv=5, scoring='accuracy')
grid_w.fit(X_w_train, y_w_train)

print(f"Best parameters: {grid_w.best_params_}")
print(f"Best CV accuracy: {grid_w.best_score_:.3f}")
print(f"Test accuracy:    {grid_w.score(X_w_test, y_w_test):.3f}")

## Exercise 2: The Effect of Scaling on KNN

1. Using the same Wine data, train a KNN pipeline **without** a scaler (just KNN directly).
2. Compare the test accuracy to the pipeline **with** StandardScaler from Exercise 1.
3. Why does scaling matter so much for this dataset? (Hint: look at the feature ranges with `wine.data.min(axis=0)` and `wine.data.max(axis=0)`.)

In [ ]:
# Write your code here


In [ ]:
#@title Click to reveal solution
# Without scaling
knn_no_scale = KNeighborsClassifier(n_neighbors=grid_w.best_params_['knn__n_neighbors'],
                                     weights=grid_w.best_params_['knn__weights'])
knn_no_scale.fit(X_w_train, y_w_train)
acc_no_scale = knn_no_scale.score(X_w_test, y_w_test)

# With scaling (from Exercise 1)
acc_scaled = grid_w.score(X_w_test, y_w_test)

print(f"Test accuracy WITHOUT scaling: {acc_no_scale:.3f}")
print(f"Test accuracy WITH scaling:    {acc_scaled:.3f}")
print()

# Look at feature ranges
print("Feature ranges:")
for i, name in enumerate(wine.feature_names):
    print(f"  {name:30s}  min={wine.data[:, i].min():.1f}  max={wine.data[:, i].max():.1f}")

print("\nFeatures like 'proline' (range 278-1680) dominate distance over")
print("features like 'nonflavanoid_phenols' (range 0.13-0.66) without scaling.")

## Exercise 3: K-Means on the Iris Dataset

1. Scale the Iris features with `StandardScaler`.
2. Run K-Means for K=2 through K=8. For each K, record the inertia and silhouette score.
3. Plot both the elbow curve and the silhouette scores.
4. Which K would you choose? Does it match the true number of species (3)?

In [ ]:
# Write your code here


In [ ]:
#@title Click to reveal solution
X_iris_scaled = StandardScaler().fit_transform(iris.data)

k_range = range(2, 9)
inertias_iris = []
sil_iris = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_iris_scaled)
    inertias_iris.append(km.inertia_)
    sil_iris.append(silhouette_score(X_iris_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias_iris, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')
axes[0].set_xticks(list(k_range))

axes[1].plot(k_range, sil_iris, 'go-', linewidth=2, markersize=8)
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')
axes[1].set_xticks(list(k_range))
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(sil_iris)]
print(f"Best K by silhouette: {best_k}")
print(f"True number of species: 3")
print(f"\nThe silhouette score often suggests K=2 for Iris because versicolor")
print(f"and virginica overlap significantly. This is a good reminder that")
print(f"clustering metrics don't always agree with ground truth labels.")

## Exercise 4: Comparing KNN and K-Means on the Same Data

This exercise highlights the difference between supervised and unsupervised approaches.

1. Use the Iris dataset (scaled). Run K-Means with K=3 and get the cluster labels.
2. Train a KNN classifier (K=5, with scaling pipeline) on the true Iris labels. Get predictions on the test set.
3. Compare: How well do the K-Means cluster assignments match the true species labels? (Use `classification_report` with the K-Means labels vs true labels.)
4. How does KNN's test accuracy compare?

In [ ]:
# Write your code here


In [ ]:
#@title Click to reveal solution
from sklearn.metrics import adjusted_rand_score

X_iris, y_iris = load_iris(return_X_y=True)

# K-Means (unsupervised)
X_iris_sc = StandardScaler().fit_transform(X_iris)
km_iris = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = km_iris.fit_predict(X_iris_sc)

# Note: K-Means labels are arbitrary (0,1,2 don't match species 0,1,2)
# Use Adjusted Rand Index to compare
ari = adjusted_rand_score(y_iris, km_labels)
print(f"K-Means Adjusted Rand Index vs true labels: {ari:.3f}")
print(f"(1.0 = perfect match, 0.0 = random)\n")

# KNN (supervised)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=5))
])
knn_pipe.fit(X_tr, y_tr)
knn_acc = knn_pipe.score(X_te, y_te)
print(f"KNN test accuracy: {knn_acc:.3f}")
print(f"\nKNN with labels outperforms K-Means without labels, as expected.")
print(f"But K-Means gets surprisingly close without ever seeing any labels!")